# Fig6 Notebook

# Load modules 

In [ ]:
from pathlib import Path
import sys
import importlib
import math

import anndata as an
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import squidpy as sq
from matplotlib.cm import ScalarMappable
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.patches import Patch
from mpl_toolkits.axes_grid1 import make_axes_locatable

plt.rcdefaults()

code_dir = "/projects/Pregnancy_placenta/Code_Publication/"
HELPER_FUNCS_DIR = Path(code_dir) / "Functions"
TMA_PATH = Path(
    "/orcd/pool/003/plopez/Projects/Preg_placenta_VISIUM/data/Final_Objs/final_TMA_obj_V2.h5ad"
)
FINAL_FIG_DIR = Path(
    "/home/plopez/orcd/pool/Projects/Preg_placenta_VISIUM/results/final_figures"
)

LIBRARY_ID = "TMA4"
DEFAULT_CROP = (1450, 1725, 1520, 1800)
SCALE_BAR_UM = 100


## load functions 

In [ ]:
if str(HELPER_FUNCS_DIR) not in sys.path:
    sys.path.append(str(HELPER_FUNCS_DIR))

import vis_tcell_helpful_func as vis_tcr_plt
importlib.reload(vis_tcr_plt)

import visium_plotting as vis_plt
importlib.reload(vis_plt)   # if you edit it and want to refresh

vis_plt.configure_publication_fonts()


# Load obj 

## load full obj 

In [ ]:

TMA_obj = an.read_h5ad(TMA_PATH)


In [ ]:
LIB = vis_plt.get_spatial_library(TMA_obj)
MICRONS_PER_PIXEL = vis_plt.get_microns_per_pixel(TMA_obj)
SCALE_BAR_PX = vis_plt.get_scale_bar_px(TMA_obj, scale_bar_um=SCALE_BAR_UM)



## load subset of TMA 

In [ ]:
# Subset the TMA object to the field of view used throughout Figure 6.
x_min, y_min, x_max, y_max = DEFAULT_CROP
spatial_coords = TMA_obj.obsm["spatial"]

mask = (
    (spatial_coords[:, 0] > x_min)
    & (spatial_coords[:, 0] < x_max)
    & (spatial_coords[:, 1] > y_min)
    & (spatial_coords[:, 1] < y_max)
)

TMA_cropped = TMA_obj[mask, :].copy()
TMA_cropped = TMA_cropped[
    ~TMA_cropped.obs["dominant_parent_type"].isin(["HBC", "NK"]), :
].copy()


# FIgure 6G : Identifying High-Tcell in TMA4

In [ ]:
inhibit_genes = ["Tcells"]
crop = (1100, 1610, 1610, 2110)
rects = [((1450, 1725, 1520, 1800), "#FF0000")]

ncols = 1
nrows = math.ceil(len(inhibit_genes) / ncols)

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(4 * ncols, 4 * nrows),
    squeeze=False,
    gridspec_kw={"wspace": 0.01, "hspace": 0.10},
)
axes = axes.ravel()

for ax, feat in zip(axes, inhibit_genes):
    sq.pl.spatial_scatter(
        TMA_obj,
        color=feat,
        library_key="library_id",
        library_id=LIBRARY_ID,
        cmap="YlOrRd",
        shape="square",
        alpha=1,
        size=2,
        outline=False,
        crop_coord=crop,
        use_raw=False,
        ax=ax,
        title="T-Cell Proportion",
    )
    vis_plt.add_scale_bar(
        ax,
        SCALE_BAR_PX,
        label=f"{SCALE_BAR_UM} µm",
        pad=0.3,
        bbox_to_anchor=None,
        bbox_transform=None,
        size_vertical=4,
    )
    vis_plt.add_rectangles(ax, rects, crop)
    vis_plt.clean_spatial_axis(ax, "T-Cell Proportion")

for ax in axes[len(inhibit_genes):]:
    ax.axis("off")

plt.tight_layout()
vis_plt.save_figure(fig, "fig6G_Tcell_proportion_V4.pdf", FINAL_FIG_DIR)
plt.show()


# Figure 6H: mMac, Tcells and dS

In [ ]:
crop = DEFAULT_CROP
features = ["mMac", "dS", "Tcells"]

fig, axes = plt.subplots(
    1,
    3,
    figsize=(12, 4),
    gridspec_kw={"wspace": 0.01, "hspace": 0.10},
)
axes = axes.ravel()

for ax, feat in zip(axes, features):
    sq.pl.spatial_scatter(
        TMA_obj,
        color=feat,
        library_key="library_id",
        library_id=LIBRARY_ID,
        cmap="YlOrRd",
        shape="square",
        size=1,
        img=True,
        outline=False,
        crop_coord=crop,
        use_raw=False,
        ax=ax,
        title="",
    )
    vis_plt.add_scale_bar(ax, SCALE_BAR_PX)
    vis_plt.clean_spatial_axis(ax, feat)

plt.subplots_adjust(left=0.03, right=0.98, top=0.90, bottom=0.08, wspace=0.22)
vis_plt.save_figure(fig, "fig6H_Tcells-CD4-CD8_mMac_dS_V5.pdf", FINAL_FIG_DIR)
plt.show()


# Figure 6I: look into ISG distribution

## generate score 

In [ ]:
isg_genes_V2 = ["CXCL9", "CXCL10", "CXCL11", "IDO1", "GBP5", "GBP4"]

vis_tcr_plt.signature_score_per_cell_min_detected(
    data=TMA_cropped,
    gene_set=isg_genes_V2,
    score_title="ISG_score_T2",
    
    min_detected_genes=2,
    expr_threshold=0,
    
    fill_value=np.nan
)

## Sup Fig 10K- Plot genes used to generate distribution

In [ ]:
crop = DEFAULT_CROP
features = isg_genes_V2

fig, axes = plt.subplots(
    2,
    3,
    figsize=(11, 8),
    gridspec_kw={"wspace": 0.01, "hspace": 0.10},
)
axes = axes.ravel()

for ax, feat in zip(axes, features):
    sq.pl.spatial_scatter(
        TMA_obj,
        color=feat,
        library_key="library_id",
        library_id=LIBRARY_ID,
        cmap="YlOrRd",
        shape="square",
        size=1,
        img=True,
        outline=False,
        crop_coord=crop,
        use_raw=False,
        ax=ax,
        title="",
    )
    vis_plt.add_scale_bar(ax, SCALE_BAR_PX)
    vis_plt.clean_spatial_axis(ax, feat)

plt.subplots_adjust(left=0.03, right=0.98, top=0.90, bottom=0.08, wspace=0.22)
vis_plt.save_figure(fig, "Supfig7K_ISG_genes_V2.pdf", FINAL_FIG_DIR)
plt.show()


## Fig 6I - Plot ISG Distribution 

In [ ]:
isg_cmap = LinearSegmentedColormap.from_list(
    "isg_cmap",
    ["#F3F4FF", "#9DB2FF", "#355CDE"],
)

crop = DEFAULT_CROP
vmax = TMA_cropped.obs["ISG_score_T2"].quantile(0.97)

fig, ax = plt.subplots(figsize=(4, 4))

sq.pl.spatial_scatter(
    TMA_cropped,
    color="ISG_score_T2",
    library_key="library_id",
    library_id=LIBRARY_ID,
    crop_coord=crop,
    cmap=isg_cmap,
    vmin=0,
    vmax=vmax,
    na_color="#F2F2F2",
    shape="square",
    size=1,
    img=True,
    outline=False,
    use_raw=False,
    ax=ax,
    colorbar=True,
)

vis_plt.add_scale_bar(ax, SCALE_BAR_PX)
vis_plt.clean_spatial_axis(ax, "ISG Score", pad=3)
vis_plt.save_figure(fig, "fig6I_ISG_score_V3.pdf", FINAL_FIG_DIR)
plt.show()


# Fig6J + SupFigs - ISG dS and ISG-mMac 

## Sup Figs 10L/M- genreate barplot and piechart of the ISG distribution across the bins

### prepping the variables...

In [ ]:
score_col = "ISG_score_T2"
comp_cols = ["dS", "Tcells", "mMac"]
dominant_threshold = 0.67

plot_colors = {
    "dS": "#7B3294",
    "Tcells": "#009E73",
    "mMac": "#0072B2",
    "Mixed": "#9E9E9E",
}

cat_order1 = ["ISG_dS", "ISG_Tcell", "ISG_mMac", "ISG_mixed", "Not_ISG"]
colors1 = {
    "ISG_dS": "#7B3294",
    "ISG_Tcell": "#009E73",
    "ISG_mMac": "#0072B2",
    "ISG_mixed": "#9E9E9E",
    "Not_ISG": "#F2F2F2",
}
label_map = {"dS": "ISG_dS", "Tcells": "ISG_Tcell", "mMac": "ISG_mMac"}

obs_all = TMA_cropped.obs.copy()
total_bins = obs_all.shape[0]

comp_all = obs_all[comp_cols].fillna(0).clip(lower=0)
row_sums_all = comp_all.sum(axis=1)
comp_frac_all = comp_all.div(row_sums_all.replace(0, np.nan), axis=0).fillna(0)

dominant_celltype_all = comp_frac_all.idxmax(axis=1)
dominant_fraction_all = comp_frac_all.max(axis=1)
isg_pos_mask = np.isfinite(obs_all[score_col]) & (obs_all[score_col] > 0)

labels1 = pd.Series("Not_ISG", index=obs_all.index, dtype="object")
is_dominant_mask = isg_pos_mask & (dominant_fraction_all > dominant_threshold)
labels1.loc[isg_pos_mask & ~is_dominant_mask] = "ISG_mixed"
labels1.loc[is_dominant_mask] = dominant_celltype_all.loc[is_dominant_mask].map(label_map)

TMA_cropped.obs["ISG_dominant_celltype"] = pd.Categorical(
    labels1,
    categories=cat_order1,
    ordered=False,
)
TMA_cropped.uns["ISG_dominant_celltype_colors"] = [colors1[c] for c in cat_order1]

obs_pos = obs_all.loc[isg_pos_mask, [score_col] + comp_cols].copy()
n_isg_pos = obs_pos.shape[0]
obs_pos = obs_pos.sort_values(score_col).reset_index(drop=True)

comp = obs_pos[comp_cols].fillna(0).clip(lower=0)
row_sums = comp.sum(axis=1)
comp_frac = comp.div(row_sums.replace(0, np.nan), axis=0).fillna(0)

dominant_fraction = comp_frac.max(axis=1)
dominant_short = comp_frac.idxmax(axis=1)
obs_pos["dominant_short"] = dominant_short.where(
    dominant_fraction > dominant_threshold,
    "Mixed",
)

print(f"{n_isg_pos}/{total_bins} bins in the FOV are ISG+")
print(TMA_cropped.obs["ISG_dominant_celltype"].value_counts(dropna=False))


### Sup Fig 10L - barplot 

In [ ]:
x = np.arange(len(obs_pos))

fig, (ax_top, ax_bot) = plt.subplots(
    2,
    1,
    figsize=(11, 6),
    sharex=True,
    gridspec_kw={"height_ratios": [1, 2], "hspace": 0.08},
)

ax_top.plot(x + 0.5, obs_pos[score_col].to_numpy(), color="firebrick", lw=1.5)
ax_top.scatter(x + 0.5, obs_pos[score_col].to_numpy(), color="firebrick", s=12)
ax_top.set_ylabel("ISG score")
ax_top.set_title(f"ISG+ bins in FOV: {n_isg_pos}/{total_bins}")

bottom = np.zeros(len(comp_frac))
for col in comp_cols:
    vals = comp_frac[col].to_numpy()
    ax_bot.bar(
        x,
        vals,
        bottom=bottom,
        width=1.0,
        align="edge",
        color=plot_colors[col],
        edgecolor="black",
        linewidth=0.15,
        alpha=0.85,
        label=col,
    )
    bottom += vals

ax_bot.axhline(dominant_threshold, color="red", linestyle="--", linewidth=1.0)
ax_bot.set_ylabel("Cell-type fraction")
ax_bot.set_xlabel("ISG+ bins ordered by ISG_score_T2")
ax_bot.set_ylim(0, 1)
ax_bot.set_xlim(0, len(x))
ax_bot.set_xticks([])

ax_bot.legend(frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.13))

plt.tight_layout()
vis_plt.save_figure(fig, "SupFigL_ISG_cellprop_barplot_V2.pdf", FINAL_FIG_DIR, format="pdf", pad_inches=0.1)
plt.show()


### Sup Fig 10M piechart 

In [ ]:
dominant_counts = obs_pos["dominant_short"].value_counts()
pie_order = ["dS", "Tcells", "mMac", "Mixed"]
pie_labels = [label for label in pie_order if label in dominant_counts.index]
pie_sizes = [dominant_counts[label] for label in pie_labels]
pie_colors = [plot_colors[label] for label in pie_labels]


def autopct_with_counts(values):
    total = np.sum(values)

    def _fmt(pct):
        count = int(round(pct * total / 100.0))
        return f"{pct:.1f}%\n(n={count})"

    return _fmt


fig2, ax2 = plt.subplots(figsize=(5, 5))
ax2.pie(
    pie_sizes,
    labels=pie_labels,
    colors=pie_colors,
    startangle=90,
    counterclock=False,
    autopct=autopct_with_counts(pie_sizes),
    wedgeprops={"edgecolor": "black", "linewidth": 0.6},
    textprops={"fontsize": 10},
)
ax2.set_title(f"Dominant cell type among ISG+ bins\n(threshold > {dominant_threshold})")

plt.tight_layout()
vis_plt.save_figure(fig2, "SupFig6M_ISG_cellprop_dominant_piechart_V2.pdf", FINAL_FIG_DIR, format="pdf", pad_inches=0.1)
plt.show()


## Fig 6J - look at ISG dominant celltyoe in field of view...

In [ ]:
crop = DEFAULT_CROP

fig, ax = plt.subplots(figsize=(4, 4))

sq.pl.spatial_scatter(
    TMA_cropped,
    color="ISG_dominant_celltype",
    library_key="library_id",
    library_id=LIBRARY_ID,
    shape="square",
    size=1,
    crop_coord=crop,
    img=True,
    outline=False,
    ax=ax,
    title="",
)

vis_plt.add_scale_bar(ax, SCALE_BAR_PX)
vis_plt.clean_spatial_axis(ax, "ISG Dominant Cell Score")

handles = [
    Patch(facecolor="#7B3294", edgecolor="none", label="ISG_dS"),
    Patch(facecolor="#009E73", edgecolor="none", label="ISG_Tcell"),
    Patch(facecolor="#0072B2", edgecolor="none", label="ISG_mMac"),
    Patch(facecolor="#9E9E9E", edgecolor="none", label="ISG_mixed"),
    Patch(facecolor="#F2F2F2", edgecolor="black", label="Not_ISG"),
]

ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.01), ncol=2, frameon=False)

plt.subplots_adjust(left=0.02, right=0.78, top=0.90, bottom=0.05)
vis_plt.save_figure(fig, "Fig6J_ISG_dom_cell_score_V3.pdf", FINAL_FIG_DIR)
plt.show()


# Fig6 M/N Plot Ligand - Receptor : mMac + Tcells 

## Fig6 M: FOV focus on ISG-mMAc and non-ISG-Mac Tcell proportion


In [ ]:
adata = TMA_cropped.copy()
library_id = LIBRARY_ID
crop_coord = DEFAULT_CROP
size = 1
source_col = "ISG_dominant_celltype"

adata.obs["Tcells_not_ISG_mMac"] = np.nan
not_mmac_mask = adata.obs[source_col].astype(str).ne("ISG_mMac")
adata.obs.loc[not_mmac_mask, "Tcells_not_ISG_mMac"] = adata.obs.loc[not_mmac_mask, "Tcells"]

adata.obs["ISG_mMac_only"] = adata.obs[source_col].astype("object")
adata.obs.loc[adata.obs["ISG_mMac_only"] != "ISG_mMac", "ISG_mMac_only"] = np.nan

cat_order = ["ISG_mMac"]
cat_colors = ["#0072B2"]
adata.obs["ISG_mMac_only"] = pd.Categorical(adata.obs["ISG_mMac_only"], categories=cat_order)
adata.uns["ISG_mMac_only_colors"] = cat_colors

fig, ax = plt.subplots(figsize=(5, 5))

sq.pl.spatial_scatter(
    adata,
    color="Tcells_not_ISG_mMac",
    library_key="library_id",
    library_id=library_id,
    crop_coord=crop_coord,
    shape="square",
    size=size,
    img=True,
    cmap="YlOrRd",
    vmin=0,
    vmax=0.8,
    na_color=(0, 0, 0, 0),
    ax=ax,
)

sq.pl.spatial_scatter(
    adata,
    color="ISG_mMac_only",
    library_key="library_id",
    library_id=library_id,
    crop_coord=crop_coord,
    shape="square",
    alpha=0.7,
    size=size,
    img=False,
    na_color=(0, 0, 0, 0),
    ax=ax,
)

ax.set_facecolor((0, 0, 0, 0))
ax.invert_yaxis()
ax.set_title("ISG_mMac + Tcell prop\nof non-ISG_mMac bins", fontsize=18, fontweight="bold", pad=4)

leg = ax.get_legend()
if leg is not None:
    leg.remove()

for extra_ax in fig.axes[1:]:
    if extra_ax is not ax:
        extra_ax.remove()

fig.subplots_adjust(bottom=0.18, right=0.82)

divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.08)

sm = ScalarMappable(norm=Normalize(vmin=0, vmax=0.8), cmap="YlOrRd")
sm.set_array([])
cbar = fig.colorbar(sm, cax=cax)
cbar.set_label("Tcell Prop (non-ISG_mMac)", rotation=270, labelpad=18)

handles = [
    Patch(facecolor="none", edgecolor="black", linestyle="--", linewidth=2, label="FOV-1"),
    Patch(facecolor="none", edgecolor="#7B3294", linestyle="--", linewidth=2, label="FOV-2"),
    Patch(facecolor="#0072B2", edgecolor="none", label="ISG_mMac"),
]
ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.01), ncol=2, frameon=False)

ax.set_xlabel("")
ax.set_ylabel("")
ax.set_xticks([])
ax.set_yticks([])

rects = [
    ((1469, 1745, 1483, 1752), "black"),
    ((1484, 1736, 1500, 1743), "#7B3294"),
]
vis_plt.add_rectangles(ax, rects, crop_coord)
vis_plt.add_scale_bar(ax, SCALE_BAR_PX, zorder=30)

vis_plt.save_figure(fig, "Fig6M_ISG-mMac_TcellProp_with_rect_V2.pdf", FINAL_FIG_DIR, format="pdf")
plt.show()


## Fig 6N - plot the genes 

### adjust box sizes 

In [ ]:
rects = [
    ((1469, 1745, 1483, 1752), "#009E73"),
    ((1484, 1736, 1500, 1743), "#7B3294"),
]

In [ ]:
inhibit_genes = ["NECTIN2", "TIGIT", "LGALS9", "HAVCR2"]
crop = DEFAULT_CROP

ncols = 2
nrows = math.ceil(len(inhibit_genes) / ncols)

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(4 * ncols, 4 * nrows),
    squeeze=False,
    gridspec_kw={"wspace": 0.10, "hspace": 0.10},
)
axes = axes.ravel()

for ax, feat in zip(axes, inhibit_genes):
    sq.pl.spatial_scatter(
        TMA_cropped,
        color=feat,
        library_key="library_id",
        library_id=LIBRARY_ID,
        cmap="YlOrRd",
        shape="square",
        alpha=1,
        size=1,
        outline=False,
        crop_coord=crop,
        use_raw=False,
        ax=ax,
        title="",
    )
    vis_plt.add_rectangles(ax, rects, crop)
    vis_plt.add_scale_bar(ax, SCALE_BAR_PX, zorder=30)
    vis_plt.clean_spatial_axis(ax, feat)

for ax in axes[len(inhibit_genes):]:
    ax.axis("off")

plt.tight_layout()
vis_plt.save_figure(fig, "Fig6N_CoInhibitor_interactions_V2.pdf", FINAL_FIG_DIR, format="pdf")
plt.show()
